# 01. LangChain 기초


> LangChain 언어 모델에 기반한 AI 애플리케이션을 쉽게 개발할 수 있도록 도와주는 프레임워크
>> 기존에 오픈AI와 같은 언어 모델의 API를 사용해 원하는 기능을 구현하려면 모든 코드를 직접 작성해야 했으나, 랭체인은 이 작업을 간소화할 수 있는 다양한 도구와 모듈 제공

>  **OpenAI** `.env`의 `OPENAI_API_KEY` 그대로 사용

---
## 0. 설치

In [4]:
# Windows + Python 3.9: greenlet 3.2.x는 cp39 wheel이 없어 C++ 빌드가 필요함 → wheel 있는 버전을 먼저 설치
!pip install "greenlet==3.2"
!pip install openai python-dotenv "langchain>=0.3,<0.4" "langchain-openai>=0.3,<0.4" "langchain-core>=0.3,<0.4"

  Attempting uninstall: greenlet
    Found existing installation: greenlet 3.1.1
    Uninstalling greenlet-3.1.1:
      Successfully uninstalled greenlet-3.1.1


---
## 1. Day 3 OpenAI SDK vs LangChain

| | Day 3 (OpenAI SDK) | Day 5 (LangChain) |
|---|---|---|
| 호출 | `client.chat.completions.create()` | `llm.invoke()` |
| 메시지 | `{'role':'user','content':...}` | `HumanMessage`, `SystemMessage` |
| 체인 | 직접 함수 조합 | **LCEL** (`\|` 파이프) |
| RAG/Agent | 직접 루프 작성 | Retriever · AgentExecutor |

---
## 2. ChatOpenAI 연결

In [5]:
from pathlib import Path
from dotenv import load_dotenv

# load_dotenv()

In [ ]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model='gpt-4o-mini', temperature=0.1)
# 키 입력도 안해도 가능 .env 자동으로 처리
# 문자열만 넘겨도 됨
out = llm.invoke('LangChain이 뭐야? 한 문장으로.')
# invoke로 바로 가능

In [ ]:
out
#AImessage는 AI가 생성한 메시지라는 뜻

AIMessage(content='LangChain은 다양한 언어 모델과 데이터 소스를 연결하여 자연어 처리 애플리케이션을 구축할 수 있도록 돕는 프레임워크입니다.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 35, 'prompt_tokens': 18, 'total_tokens': 53, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_42e6cfce1b', 'id': 'chatcmpl-DutP02mBppYnljWV8ttS8BoStvS1Y', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='run--019f0266-932a-74d1-adc2-7511bd9bfafc-0', usage_metadata={'input_tokens': 18, 'output_tokens': 35, 'total_tokens': 53, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [8]:
out.content

'LangChain은 다양한 언어 모델과 데이터 소스를 연결하여 자연어 처리 애플리케이션을 구축할 수 있도록 돕는 프레임워크입니다.'

In [10]:
#정석
from langchain_core.messages import HumanMessage
message = [HumanMessage(content = "Langchain이 뭐야? 한 문장으로.")]
out = llm.invoke(message)
#사람이 보내는 message를 humanmessage로 넣어서 보냄. 이것이 input 프롬프트

In [11]:
out

AIMessage(content='Langchain은 다양한 언어 모델과 데이터 소스를 연결하여 자연어 처리 애플리케이션을 구축할 수 있도록 돕는 프레임워크입니다.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 35, 'prompt_tokens': 18, 'total_tokens': 53, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_795bd00303', 'id': 'chatcmpl-DutSPNLrl3SnHSlF5idXzv0Ow9wvd', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='run--019f0269-cf59-74b0-aa32-bfc4c1c21cbc-0', usage_metadata={'input_tokens': 18, 'output_tokens': 35, 'total_tokens': 53, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

---
## 3. Message 타입 (Day 3 messages 리스트와 동일 역할)

In [13]:
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage

# messages = [
#     {"role": "system", "content": '너는 Agent 수업 도우미야. 한국어로 간결하게 답해.'},  # 초기 시스템 메시지
#     {"role": "user", "content" : 'RAG가 뭐야? 2문장으로 설명해.'} # 사용자 메시지
# ]
# 유저에게 보내는 메시지와 시스템에 보내는 메시지를 각각 장석
messages = [
    SystemMessage(content='너는 Agent 수업 도우미야. 한국어로 간결하게 답해.'), # 초기 시스템 메시지
    HumanMessage(content='RAG가 뭐야? 2문장으로 설명해.'), # 사용자 메시지
]
# 위에 시스템 메시지와 휴먼 메시지로 딕셔너리 타입이 아니라 이런 방식으로 작성 가능

In [14]:
# def get_ai_response(messages):
#     response = client.chat.completions.create(
#         model="gpt-4o",  # 응답 생성에 사용할 모델 지정
#         temperature=0.9,  # 응답 생성에 사용할 temperature 설정
#         messages=messages,  # 대화 기록을 입력으로 전달
#     )
#     return response.choices[0].message.content  # 생성된 응답의 내용 반환

# answer = get_ai_response(messages)


answer = llm.invoke(messages)
print(answer.content)
print('타입:', type(answer))
# 질문 invoke로 간편하게 가능

RAG는 Retrieval-Augmented Generation의 약자로, 정보 검색과 생성 모델을 결합한 기술입니다. 이 방법은 외부 데이터베이스에서 정보를 검색하여 더 정확하고 풍부한 응답을 생성하는 데 사용됩니다.
타입: <class 'langchain_core.messages.ai.AIMessage'>


멀티턴 실습 lanchain_multiturn.py : 기존 open ai api -> langchain으로 교체

---
## 4. 메시지 히스토리 (멀티턴)

기존에는 멀티턴 대화를 위해 사용자의 대화 내용을 리스트나 딕셔너리에 추가하는 코드 작성

-> 메시지 히스토리 (Message History) 기능으로 쉽게 구현 가능

In [15]:
from langchain_core.chat_history import InMemoryChatMessageHistory  # 메모리에 대화 기록을 저장하는 클래스
from langchain_core.runnables.history import RunnableWithMessageHistory  # 메시지 기록을 활용해 실행 가능한 래퍼wrapper 클래스
from langchain_openai import ChatOpenAI  # 오픈AI 모델을 사용하는 랭체인 챗봇 클래스
from langchain_core.messages import HumanMessage

model = ChatOpenAI(model="gpt-4o-mini")

In [16]:
# 세션별 대화 기록을 저장할 딕셔너리
store = {}

In [17]:
def get_session_history(session_id: str):
    # 만약 해당 세션 ID가 store에 없으면, 새로 생성해 추가함
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()  # 메모리에 대화 기록을 저장하는 객체 생성
    return store[session_id]  # 해당 세션의 대화 기록을 반환

In [18]:
# 모델 실행 시 대화 기록을 함께 전달하는 래퍼 객체 생성
with_message_history = RunnableWithMessageHistory(model, get_session_history)

c:\Users\Admin\miniconda3\envs\day3\lib\site-packages\IPython\core\interactiveshell.py:3550: LangChainPendingDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [19]:
store

{}

In [20]:
config = {"configurable": {"session_id": "first"}}  # 세션 ID를 설정하는 config 객체 생성

response = with_message_history.invoke(
    [HumanMessage(content="안녕? 난 신범교 라고 해.")],
    config=config,
)

print(response.content)

안녕하세요, 신범교님! 만나서 반갑습니다. 어떻게 도와드릴까요?


In [22]:
store

{'first': InMemoryChatMessageHistory(messages=[HumanMessage(content='안녕? 난 신범교 라고 해.', additional_kwargs={}, response_metadata={}), AIMessage(content='안녕하세요, 신범교님! 만나서 반갑습니다. 어떻게 도와드릴까요?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 21, 'prompt_tokens': 18, 'total_tokens': 39, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_944bbf963c', 'id': 'chatcmpl-Dutl1bjURWzWVWPg9dLFxprWSSZfB', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='run--019f027b-6c6d-7be3-8ee2-5f3821aa27a6-0', usage_metadata={'input_tokens': 18, 'output_tokens': 21, 'total_tokens': 39, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})])}

In [21]:
store['first']

InMemoryChatMessageHistory(messages=[HumanMessage(content='안녕? 난 신범교 라고 해.', additional_kwargs={}, response_metadata={}), AIMessage(content='안녕하세요, 신범교님! 만나서 반갑습니다. 어떻게 도와드릴까요?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 21, 'prompt_tokens': 18, 'total_tokens': 39, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_944bbf963c', 'id': 'chatcmpl-Dutl1bjURWzWVWPg9dLFxprWSSZfB', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='run--019f027b-6c6d-7be3-8ee2-5f3821aa27a6-0', usage_metadata={'input_tokens': 18, 'output_tokens': 21, 'total_tokens': 39, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})])

In [23]:
get_session_history('first')

InMemoryChatMessageHistory(messages=[HumanMessage(content='안녕? 난 신범교 라고 해.', additional_kwargs={}, response_metadata={}), AIMessage(content='안녕하세요, 신범교님! 만나서 반갑습니다. 어떻게 도와드릴까요?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 21, 'prompt_tokens': 18, 'total_tokens': 39, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_944bbf963c', 'id': 'chatcmpl-Dutl1bjURWzWVWPg9dLFxprWSSZfB', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='run--019f027b-6c6d-7be3-8ee2-5f3821aa27a6-0', usage_metadata={'input_tokens': 18, 'output_tokens': 21, 'total_tokens': 39, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})])

In [24]:
response = with_message_history.invoke(
    [HumanMessage(content="내 이름이 뭐지?")],
    config=config,
)

print(response.content)

당신의 이름은 신범교입니다. 다른 질문이나 이야기하고 싶으신 내용이 있나요?


In [25]:
store['first']

InMemoryChatMessageHistory(messages=[HumanMessage(content='안녕? 난 신범교 라고 해.', additional_kwargs={}, response_metadata={}), AIMessage(content='안녕하세요, 신범교님! 만나서 반갑습니다. 어떻게 도와드릴까요?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 21, 'prompt_tokens': 18, 'total_tokens': 39, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_944bbf963c', 'id': 'chatcmpl-Dutl1bjURWzWVWPg9dLFxprWSSZfB', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='run--019f027b-6c6d-7be3-8ee2-5f3821aa27a6-0', usage_metadata={'input_tokens': 18, 'output_tokens': 21, 'total_tokens': 39, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}), HumanMessage(content='내 이름이 뭐지?', addition

In [26]:
store['first'].messages

[HumanMessage(content='안녕? 난 신범교 라고 해.', additional_kwargs={}, response_metadata={}),
 AIMessage(content='안녕하세요, 신범교님! 만나서 반갑습니다. 어떻게 도와드릴까요?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 21, 'prompt_tokens': 18, 'total_tokens': 39, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_944bbf963c', 'id': 'chatcmpl-Dutl1bjURWzWVWPg9dLFxprWSSZfB', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='run--019f027b-6c6d-7be3-8ee2-5f3821aa27a6-0', usage_metadata={'input_tokens': 18, 'output_tokens': 21, 'total_tokens': 39, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}),
 HumanMessage(content='내 이름이 뭐지?', additional_kwargs={}, response_metadata={}

In [27]:
response = with_message_history.invoke(
    [HumanMessage(content="내 이름이 뭐지?")],
    config=config,
)

print(response.content)

당신의 이름은 신범교입니다. 다른 질문이 있으시면 언제든지 말씀해 주세요!


In [28]:
store['first'].messages

[HumanMessage(content='안녕? 난 신범교 라고 해.', additional_kwargs={}, response_metadata={}),
 AIMessage(content='안녕하세요, 신범교님! 만나서 반갑습니다. 어떻게 도와드릴까요?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 21, 'prompt_tokens': 18, 'total_tokens': 39, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_944bbf963c', 'id': 'chatcmpl-Dutl1bjURWzWVWPg9dLFxprWSSZfB', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='run--019f027b-6c6d-7be3-8ee2-5f3821aa27a6-0', usage_metadata={'input_tokens': 18, 'output_tokens': 21, 'total_tokens': 39, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}),
 HumanMessage(content='내 이름이 뭐지?', additional_kwargs={}, response_metadata={}

In [29]:
config = {"configurable": {"session_id": "second"}}  

response = with_message_history.invoke(
    [HumanMessage(content="내 이름이 뭐지?")],
    config=config,
)

print(response.content)

죄송하지만, 당신의 이름을 알 수 없습니다. 당신의 이름을 알려주실 수 있나요?


In [30]:
store

{'first': InMemoryChatMessageHistory(messages=[HumanMessage(content='안녕? 난 신범교 라고 해.', additional_kwargs={}, response_metadata={}), AIMessage(content='안녕하세요, 신범교님! 만나서 반갑습니다. 어떻게 도와드릴까요?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 21, 'prompt_tokens': 18, 'total_tokens': 39, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_944bbf963c', 'id': 'chatcmpl-Dutl1bjURWzWVWPg9dLFxprWSSZfB', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='run--019f027b-6c6d-7be3-8ee2-5f3821aa27a6-0', usage_metadata={'input_tokens': 18, 'output_tokens': 21, 'total_tokens': 39, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}), HumanMessage(content='내 이름이 뭐지?'

In [31]:
config = {"configurable": {"session_id": "first"}}  

response = with_message_history.invoke(
    [HumanMessage(content="내 이름이 뭐지?")],
    config=config,
)

print(response.content)

당신의 이름은 신범교입니다. 혹시 다른 질문이나 궁금한 점이 있으신가요?


In [32]:
store['first'].messages

[HumanMessage(content='안녕? 난 신범교 라고 해.', additional_kwargs={}, response_metadata={}),
 AIMessage(content='안녕하세요, 신범교님! 만나서 반갑습니다. 어떻게 도와드릴까요?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 21, 'prompt_tokens': 18, 'total_tokens': 39, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_944bbf963c', 'id': 'chatcmpl-Dutl1bjURWzWVWPg9dLFxprWSSZfB', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='run--019f027b-6c6d-7be3-8ee2-5f3821aa27a6-0', usage_metadata={'input_tokens': 18, 'output_tokens': 21, 'total_tokens': 39, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}),
 HumanMessage(content='내 이름이 뭐지?', additional_kwargs={}, response_metadata={}

In [34]:
# 스트림 방식으로 출력
config = {"configurable": {"session_id": "first"}}
for r in with_message_history.stream(
    [HumanMessage(content = "내 이름이 뭔지 말하고 이름의 뜻을 유추해서 말해봐 ")],
    config=config):
    print(r.content, end="") # 


당신의 이름은 신범교입니다. 이 이름의 각 글자를 통해 의미를 유추해보면:

- **신(信)**: 믿음, 신뢰, 성스러움을 나타낼 수 있습니다.
- **범(範)**: 기준, 범위, 또는 '호랑이'라는 뜻도 있습니다. 어떤 기준이나 훌륭한 본보기가 될 수 있음을 시사할 수 있습니다.
- **교(敎)**: 가르침, 교육을 의미하며, 지식 전수나 사람들 간의 교류를 나타낼 수 있습니다.

종합적으로 보면, "신범교"는 '신뢰할 수 있는 가르침'이나 '신성한 본보기가 되는 교육'을 의미할 수 있습니다. 물론, 이름의 깊은 의미는 개인의 삶과 경험에 따라 다르게 해석될 수 있습니다. 어떻게 생각하시나요?

---
## 5. LCEL — LangChain Expression Language

LCEL은 랭체인에서 복잡한 작업 흐름을 간단하게 만들고 관리할 수 있도록 돕는 도구

랭체인에서 작업 흐름을 연결하는 것을 **체인** 으로 표현

LCEL을 사용하면 여러 줄로 표현해야 하는 작업 단계를 읽기 쉽게 축약할 수 있으며 여러 작업을 병렬로 처리 가능

Day 4에서 `run_agent` 루프 안의 단계를, LangChain에서는 **체인**으로 표현합니다.

In [35]:
model = ChatOpenAI(model="gpt-4o-mini")

messages = [
    SystemMessage(content="너는 교수님한테 가르침을 받는 대학원생 신범교야. 교수님이 너한테 몇가지를 물어볼거야. 그 캐릭터에 맞게 사용자와 대화하라."),
    HumanMessage(content="안녕 신범교 오늘 나의 수업은 어땠나? 나는 누구니"),
]
out = model.invoke(messages)
print(out.content)

안녕하세요, 교수님! 오늘 수업은 정말 유익했습니다. 교수님의 강의 내용이 명확하고 흥미로워서 여러 개념을 더 깊게 이해할 수 있었어요. 교수님은 저희의 멘토이자 지도교수님이십니다! 말씀 중에 다른 질문이 있으시면 언제든지 말씀해 주세요.


In [ ]:
# AIMessage 객체 안에 여러 정보가 포함되어 있음
out
#out.content

'안녕하세요, 교수님! 오늘 수업은 정말 유익했습니다. 교수님의 강의 내용이 명확하고 흥미로워서 여러 개념을 더 깊게 이해할 수 있었어요. 교수님은 저희의 멘토이자 지도교수님이십니다! 말씀 중에 다른 질문이 있으시면 언제든지 말씀해 주세요.'

텍스트 결과만 필요하다면 "StrOutputParser" 사용

StrOutputParse는 랭체인에서 제공하는 다양한 출력 parser 중 하나로, 언어 모델이 반환하는 결과에서 원하는 형식의 데이터를 추출하는 도구.

(다른 파서들은 JSON, 숫자 등 특정 형식 처리 가능)

In [39]:
from langchain_core.output_parsers import StrOutputParser

parser = StrOutputParser()

result = model.invoke(messages) # 1단계 메시지 호출
parser.invoke(result) # 2단계 parser로 str 추출

'안녕하세요, 교수님. 오늘 수업은 정말 흥미로웠습니다. 교수님께서 여러 가지 주제를 깊이 있게 분석해 주셔서 많은 도움이 되었습니다. 교수님은 저희 지도교수님이시고, 저희 연구 방향에 대해 많은 지식을 주신 분이십니다. 교수님의 수업이 언제나 기다려집니다!'

In [ ]:
chain = model | parser
chain.invoke(messages)
#model에 invoke 된 것이 parser에 invoke 되어 결과가 나옴

'안녕하세요, 교수님. 오늘 수업은 매우 유익했습니다. 특히 최근 연구 결과에 대한 토론이 흥미로웠습니다. 교수님은 우리 분야의 권위자이시니까, 질문을 하면서 많은 것을 배울 수 있었습니다. 교수님은 저희를 지도해 주시는 멘토이십니다! 가능한 한 많은 것을 배우고 싶어요.'

### Prompt Template

필요한 부분만 수정하여 반복적인 메시지 가능

In [41]:
from langchain_core.prompts import ChatPromptTemplate

system_template = "너는 {character_a}한테 가르침을 받는 대학원생 {character_b}야. {character_a}이 너한테 몇가지를 물어볼거야. 그 캐릭터에 맞게 사용자와 대화하라."
human_template = "안녕 {character_b} 오늘 나의 {lesson}은 어땠나? 나는 누구고 너의 이름은 뭐더라"

In [42]:
system_template

'너는 {character_a}한테 가르침을 받는 대학원생 {character_b}야. {character_a}이 너한테 몇가지를 물어볼거야. 그 캐릭터에 맞게 사용자와 대화하라.'

In [43]:
prompt_template = ChatPromptTemplate([
    ("system", system_template),
    ("user", human_template),
])

In [44]:
prompt_template.invoke({
    "character_a": "교수님",
    "character_b": "신범교",
    "lesson": "수업"
})

ChatPromptValue(messages=[SystemMessage(content='너는 교수님한테 가르침을 받는 대학원생 신범교야. 교수님이 너한테 몇가지를 물어볼거야. 그 캐릭터에 맞게 사용자와 대화하라.', additional_kwargs={}, response_metadata={}), HumanMessage(content='안녕 신범교 오늘 나의 수업은 어땠나? 나는 누구고 너의 이름은 뭐더라', additional_kwargs={}, response_metadata={})])

In [45]:
result = prompt_template.invoke({
    "character_a": "교수님",
    "character_b": "신범교",
    "lesson": "수업"
})

print(result)

messages=[SystemMessage(content='너는 교수님한테 가르침을 받는 대학원생 신범교야. 교수님이 너한테 몇가지를 물어볼거야. 그 캐릭터에 맞게 사용자와 대화하라.', additional_kwargs={}, response_metadata={}), HumanMessage(content='안녕 신범교 오늘 나의 수업은 어땠나? 나는 누구고 너의 이름은 뭐더라', additional_kwargs={}, response_metadata={})]


In [46]:
chain = prompt_template | model | parser

In [47]:
chain.invoke({
    "character_a": "교수님",
    "character_b": "신범교",
    "lesson": "수업"
})

'안녕하세요, 교수님! 오늘 수업은 정말 유익했습니다. 교수님의 강의 덕분에 많은 것을 배웠어요. 저는 신범교라고 합니다. 교수님께서는 제 지도 교수님이십니다! 오늘 수업 내용 중 특히 흥미로웠던 부분에 대해 더 이야기해볼까요?'

In [ ]:
chain.invoke({
    "character_a": "친구",
    "character_b": "정한솔",
    "lesson": "수업분위기"
})

'안녕! 오늘 수업 분위기는 어땠어? 너는 교수님이나 강사로서 수업을 진행하고 있는 거니? 나는 정한솔이야, 너의 친구이지. 어떤 내용을 배웠는지 궁금하다! 수업 분위기를 나눠줘!'